# 3 · Literature-Grounded Originality Reports

The full framework grounds the novelty assessment in related work:

1. extract the manuscript's **core ideas, methods, contributions, data sources and experiments**;
2. query **Semantic Scholar** with each aspect and extract the same structure from the retrieved papers;
3. build a **similarity graph** and rank the manuscript's neighbours by similarity and centrality;
4. ask the Novelty Reviewer for a score and assessment **conditioned on the top-ranked prior work**.

**Requirements:** a GPU (two 8B models need ~40 GB; `share_model=True` needs ~20 GB), internet access, and ideally a
[Semantic Scholar API key](https://www.semanticscholar.org/product/api) in `S2_API_KEY`.

In [ ]:
# Install the package (run once). In Colab, clone the repository first:
# !git clone https://github.com/m2l-lab/Novelty-Reviewer && cd Novelty-Reviewer
# !pip install -q -e "Novelty-Reviewer[pipeline]"

In [ ]:
import os

from IPython.display import Markdown

from novelty_reviewer import OriginalityAnalyzer
from novelty_reviewer.pdf import pdf_to_markdown

# os.environ["S2_API_KEY"] = "..."   # optional, raises the Semantic Scholar rate limit
analyzer = OriginalityAnalyzer.from_pretrained(share_model=True)

## Analysing a manuscript

In [ ]:
manuscript = pdf_to_markdown("my_paper.pdf")   # or download_openreview_pdf(<paper_id>), see notebook 2
report = analyzer.analyze(manuscript, top_k=5)

In [ ]:
Markdown(report.to_markdown())

## Inspecting the evidence

In [ ]:
print("Search queries:")
for q in report.queries:
    print(" •", q)
print("\nExtracted manuscript knowledge:")
for field in ("core_ideas", "methods", "contributions"):
    print(f"  {field}: {report.manuscript[field][:3]}")

In [ ]:
for paper in report.related_work:
    print(f"{paper.similarity:5.1f}  {paper.knowledge['title']}  ({paper.metadata.get('year')})")
analyzer.graph.plot("knowledge_graph.png")

## Saving and tuning

`report.save("reports/")` writes markdown and JSON. Graph parameters can be changed when creating the analyzer:
`edge_threshold` (minimum similarity of an edge), `radius` (minimum similarity of a reported paper),
`similarity_weight` (similarity vs. centrality in the ranking) and `max_candidates`.

From the command line:

```bash
novelty-reviewer analyze --pdf my_paper.pdf --output reports --share-model
```

In [ ]:
print(report.save("reports"))